In [1]:
import os
import sys
import numpy as np
import pydot
import rmgpy.chemkin
import rmgpy.tools.fluxdiagram
import copy

import networkx as nx

import pickle

import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

sys.path.append(os.environ['DATABASE_DIR'])
import database_fun

import rmgpy.data.thermo

Loading DFT database from /home/moon/autoscience_workflow/database


In [2]:
# Options controlling the individual flux diagram renderings:
program = 'dot'  # The program to use to lay out the nodes and edges
max_node_count = 50  # The maximum number of nodes to show in the diagram
max_edge_count = 50  # The maximum number of edges to show in the diagram
concentration_tol = 1e-6  # The lowest fractional concentration to show (values below this will appear as zero)
species_rate_tol = 1e-6  # The lowest fractional species rate to show (values below this will appear as zero)
max_node_pen_width = 7.0  # The thickness of the border around a node at maximum concentration
max_edge_pen_width = 9.0  # The thickness of the edge at maximum species rate


In [4]:
species_image_path = './species_images'  # where to get/place the images of each species later on
os.makedirs(species_image_path, exist_ok=True)


# --------------------------------- EDIT THIS ----------------------------
mech_1_inp = f'/home/moon/propane/RMG_min_1/chem_annotated.inp'
mech_1_dict = f'/home/moon/propane/RMG_min_1/species_dictionary.txt'


# Still have to load the mechanisms to get the lists of species and reactions
species_list1, _reaction_list1 = rmgpy.chemkin.load_chemkin_file(mech_1_inp, mech_1_dict)

num_species1 = len(species_list1)


times1 = np.load(os.path.join(os.path.dirname(mech_1_inp), 'times.npy'))
concentrations1 = np.load(os.path.join(os.path.dirname(mech_1_inp), 'concentrations.npy'))
reaction_rates1 = np.load(os.path.join(os.path.dirname(mech_1_inp), 'rates.npy'))

assert concentrations1.shape[1] == num_species1

# you have to come up with your own mapping if the number of cantera reactions doesn't match the number of RMG reactions
with open(os.path.join(os.path.dirname(mech_1_inp), 'ct2rmg_rxn.pickle'), 'rb') as f:
    rxn_map1 = pickle.load(f)

assert reaction_rates1.shape[1] == len(rxn_map1)

reaction_list1 = []
for key in rxn_map1:
    reaction_list1.append(_reaction_list1[rxn_map1[key]])
print('constructed reaction_list1')


constructed reaction_list1


In [21]:
r = 1800
print(reaction_list1[r])
print('-------------------------------')
for pair in reaction_list1[r].pairs:
    
    print(pair[0], '->', pair[1])

CC(19) + C=CCO[O](273) <=> C[CH2](20) + C=CCOO(204)
-------------------------------
C=CCO[O](273) -> C=CCOO(204)
CC(19) -> C[CH2](20)


In [11]:
# Compute the rates between each pair of species (build up big matrices)
species_rates1 = np.zeros((len(times1), num_species1, num_species1), float)
for index1, reaction1 in enumerate(reaction_list1):
    rate1 = reaction_rates1[:, index1]
    if not reaction1.pairs: reaction1.generate_pairs()
    for reactant1, product1 in reaction1.pairs:
        reactant_index1 = species_list1.index(reactant1)
        product_index1 = species_list1.index(product1)
        species_rates1[:, reactant_index1, product_index1] += rate1
        species_rates1[:, product_index1, reactant_index1] -= rate1


In [29]:
times1[1000]

6.477227699257147e-05

In [56]:
t_index = 1000

FLUX_CUTOFF = 1e-15
G = nx.DiGraph()
for i in range(species_rates1.shape[1]):
    for j in range(i):
        if np.abs(species_rates1[t_index, i, j]) < FLUX_CUTOFF:
            continue


        node1 = rmgpy.chemkin.get_species_identifier(species_list1[i])
        node2 = rmgpy.chemkin.get_species_identifier(species_list1[j])

        # for reference,
        # species_rates1[:, reactant_index1, product_index1] += rate1
        if species_rates1[t_index, i, j] > 0:  # positive flux from reactants to products, node1 to node2
            G.add_edge(node1, node2, **{"total_flux": species_rates1[t_index, i, j]})
        else:
            G.add_edge(node2, node1, **{"total_flux": -species_rates1[t_index, i, j]})



In [70]:
for i in range(100):
    for p in reaction_list1[i].pairs:
        if rmgpy.chemkin.get_species_identifier(p[0]) in ['propane(1)', 'H(14)'] and \
                rmgpy.chemkin.get_species_identifier(p[1]) in ['propane(1)', 'H(14)']:
            print(i, reaction_list1[i])

41 propane(1) <=> H(14) + C[CH]C(22)
67 propane(1) <=> H(14) + [CH2]CC(23)


In [74]:
def print_pair(pair):
    print(rmgpy.chemkin.get_species_identifier(pair[0]), '->', rmgpy.chemkin.get_species_identifier(pair[1]))

In [77]:
print_pair(reaction_list1[41].pairs[0])
print_pair(reaction_list1[41].pairs[1])

propane(1) -> H(14)
propane(1) -> C[CH]C(22)


In [57]:
rmgpy.chemkin.get_species_identifier(species_list1[18])

'OH(15)'

In [58]:
src_species = 'propane(1)'
tgt_species = 'OH(15)'

paths = list(nx.all_simple_paths(G, source=src_species, target=tgt_species, cutoff=5))

In [65]:
def get_min_flux_on_path(path):
    path_fluxes = []
    for i in range(1, len(path)):
        data = G.get_edge_data(path[i-1], path[i])
        if data is None:
            data = G.get_edge_data(path[i], path[i-1])
            if data is None:
                return 0
            path_fluxes.append(-data['total_flux'])
        else:        
            path_fluxes.append(data['total_flux'])
    return np.min(path_fluxes)

all_path_fluxes = [get_min_flux_on_path(p) for p in paths]

In [109]:
CUTOFF_FRACTION = 0.9
path_flux_total = np.sum(np.abs(all_path_fluxes))
line_drawn = False
path_flux_sum = 0

indices = np.arange(len(paths))
sorted_order = [x for _, x in sorted(zip(all_path_fluxes, indices))][::-1]
for i in range(20):
    j = sorted_order[i]
    # if paths[j][1] == 'H(14)':
        # continue
    
    print(j, paths[j], all_path_fluxes[j])
    path_flux_sum += np.abs(all_path_fluxes[j])
    
    if path_flux_sum > path_flux_total * CUTOFF_FRACTION and not line_drawn:
        print('----------------------------------------------------')
        line_drawn = True
    

1539 ['propane(1)', 'C[CH]C(22)', 'H(14)', 'O(5)', 'OH(15)'] 0.8411742363186412
582 ['propane(1)', 'C[CH2](20)', 'H(14)', 'O(5)', 'OH(15)'] 0.8411742363186412
1571 ['propane(1)', 'C[CH]C(22)', 'H(14)', 'H2O2(17)', 'OH(15)'] 0.6670869129749111
614 ['propane(1)', 'C[CH2](20)', 'H(14)', 'H2O2(17)', 'OH(15)'] 0.6670869129749111
1555 ['propane(1)', 'C[CH]C(22)', 'H(14)', 'HO2(16)', 'H2O2(17)', 'OH(15)'] 0.4072764593127005
598 ['propane(1)', 'C[CH2](20)', 'H(14)', 'HO2(16)', 'H2O2(17)', 'OH(15)'] 0.4072764593127005
1554 ['propane(1)', 'C[CH]C(22)', 'H(14)', 'HO2(16)', 'OH(15)'] 0.34209906664930845
597 ['propane(1)', 'C[CH2](20)', 'H(14)', 'HO2(16)', 'OH(15)'] 0.34209906664930845
2321 ['propane(1)', '[CH2]CC(23)', '[CH3](21)', 'CO[O](32)', '[CH2]OO(94)', 'OH(15)'] 0.26938180534652784
1298 ['propane(1)', '[CH3](21)', 'CO[O](32)', '[CH2]OO(94)', 'OH(15)'] 0.26938180534652784
1370 ['propane(1)', '[CH3](21)', 'C[O](89)', 'H(14)', 'H2O2(17)', 'OH(15)'] 0.24126411969128114
1369 ['propane(1)', '[CH3

In [107]:
# for a given step, find the top reactions contributing to that flux pair

path_index = 1539
for path_step in range(len(paths[path_index]) - 1):
    print(f'                                  Step {path_step + 1}')
    node1 = paths[path_index][path_step]
    node2 = paths[path_index][path_step + 1]
    
    print('                      ', node1, '->', node2)
    print('========================================================================')

    
    my_reactions = []
    my_rates = []
    
    for index1, reaction1 in enumerate(reaction_list1):
        rate1 = reaction_rates1[t_index, index1]
        if not reaction1.pairs: reaction1.generate_pairs()
        for reactant1, product1 in reaction1.pairs:
            reactant_index1 = species_list1.index(reactant1)
            product_index1 = species_list1.index(product1)
    
            r = rmgpy.chemkin.get_species_identifier(species_list1[reactant_index1])
            p = rmgpy.chemkin.get_species_identifier(species_list1[product_index1])
            if r == node1 and p == node2:
                my_reactions.append(index1)
                my_rates.append(rate1)
                # print(index1, reaction_list1[index1], rate1)
            elif r == node2 and p ==node1:
                my_reactions.append(index1)
                my_rates.append(-rate1)
                # print(index1, reaction_list1[index1], -rate1)
    
    indices = np.arange(len(my_reactions))
    sorted_order = [x for _, x in sorted(zip(my_rates, indices))][::-1]
    
    CUTOFF_FRACTION = 0.9
    flux_total = np.sum(np.abs(my_rates))
    flux_sum = 0
    line_drawn = False
    for i in range(min(10, len(my_reactions))):
        
        j = sorted_order[i]
    
        reaction_index = ''
        if not line_drawn:
            reaction_index = str(database_fun.get_unique_reaction_index(reaction_list1[my_reactions[j]]))
        
        print(reaction_index, reaction_list1[my_reactions[j]], my_rates[j])
        flux_sum += np.abs(my_rates[j])
    
        if flux_sum > flux_total * CUTOFF_FRACTION and not line_drawn:
            print('--------------------------------------------------------------------')
            line_drawn = True

        if line_drawn:
            break
    print()
    print()

                                  Step 1
                       propane(1) -> C[CH]C(22)
28941 OH(15) + propane(1) <=> H2O(8) + C[CH]C(22) 1.0039901110799823
28940 H2(13) + C[CH]C(22) <=> H(14) + propane(1) 0.8036855179314767
23687 CH4(10) + C[CH]C(22) <=> [CH3](21) + propane(1) 0.19171968410772414
--------------------------------------------------------------------


                                  Step 2
                       C[CH]C(22) -> H(14)
380 H(14) + C3H6(12) <=> C[CH]C(22) 1.8610746219908085
--------------------------------------------------------------------


                                  Step 3
                       H(14) -> O(5)
0 O2(2) + H(14) <=> O(5) + OH(15) 0.8730976092097006
--------------------------------------------------------------------


                                  Step 4
                       O(5) -> OH(15)
33017 O(5) + propane(1) <=> OH(15) + [CH2]CC(23) 0.7488925858532689
491 O(5) + CC(19) <=> OH(15) + C[CH2](20) 0.12898889689169482
126 O(5)